## Merge Author Profiles — Rich Name + Institution/Coauthor Signal

Pairs no-ORCID profiles with the same dominant parsed name (first ≥ 2, middle ≥ 3, last ≥ 2) where at least one precision signal fires — institution_id overlap OR ≥ 1 shared coauthor on a published work. Companion to `MergeAuthorsOrcidCollision`: ORCID handled fragmented profiles with claimed ORCIDs; this pass handles the much larger no-ORCID universe.

**Scope:** size-2 rich-name groups in the no-ORCID universe. Audit (2026-04-29) found ~165K pairs with institution overlap (~96% precision), ~150K with coauthor overlap but no inst overlap (~100% precision), ~71K with both. Combined ~328K candidate pairs, ~330K losers expected.

**Precision guards:**
1. **Rich name only.** Dominant parsed `first ≥ 2`, `middle ≥ 3`, `last ≥ 2`. Middle ≥ 3 is the strongest precision lever short of ORCID — collisions on (first, last) alone are too common to merge safely with this rule (Wei Wang, Jason Priem class).
2. **Dominant raw_name majority (≥ 50%).** Per profile, the dominant `raw_author_name` is weighted by `work_authors` row count and must represent ≥ 50% of the profile's `work_authors` rows. Drops profiles where stray contaminating raw_names compete with the canonical (a Phase 1 audit surfaced a "William Rappard" profile with a stray `Oppenheimer Hallett Carr` raw_name from polluted attribution; the array-EXPLODE-equal-weight pattern in Phase 1 was vulnerable to this).
3. **Group size = 2.** Pairs are the cleanest tier (audit ~96–100%); larger same-parsed-name groups have a long tail of common-name false positives.
4. **Sink cap.** Drop pairs containing any profile `works_count > 5,000` (matches Phase 1 — pollution-sink avoidance).
5. **Precision signal required:** at least one of —
   - **Institution overlap.** Any institution_id common to both profiles' `last_known_institutions` or `affiliations`. Audit precision ~96%.
   - **Coauthor overlap ≥ 1.** Any other `author_id` co-listed with both pair members on at least one work apiece. Audit precision ~100%.

**Winner selection:** Per pair, max `works_count` → oldest `created_date` → lowest `id` (deterministic; same as Phase 1).

**Loser handling:** *No DELETE.* Following Phase 1's reversed-DELETE learning, this pass rewrites `work_authors.author_id` from loser → winner only; `CreateAuthors` drains the loser's `works_count` to 0 on the next refresh, so Elasticsearch sees a zero-works document instead of a missing-id orphan.

**Rollback plan:** No automatic rollback. Audit log `openalex.authors.author_merge_log_richname` captures all pair metadata and signal flags; sufficient to reconstruct any loser via `STILL_UNMATCHED` + `MatchAuthors`.

### Cell 1: Build merge candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.merge_candidates_richname AS
WITH
no_orcid_profiles AS (
  SELECT id AS author_id, full_name, display_name, block_key,
         works_count, cited_by_count, created_date,
         TRANSFORM(COALESCE(last_known_institutions, ARRAY()), x -> x.id) AS lki_ids,
         TRANSFORM(COALESCE(affiliations,            ARRAY()), x -> x.institution.id) AS aff_ids
  FROM openalex.authors.openalex_authors
  WHERE (orcid IS NULL OR orcid = '')
),
ra_counts AS (
  -- Per (profile, raw_name): work_authors row count
  SELECT wa.author_id, wa.raw_author_name, COUNT(*) AS n_works
  FROM openalex.works.work_authors wa
  JOIN no_orcid_profiles n ON n.author_id = wa.author_id
  WHERE wa.raw_author_name IS NOT NULL AND TRIM(wa.raw_author_name) <> ''
  GROUP BY wa.author_id, wa.raw_author_name
),
ra_dominant AS (
  -- Dominant raw_name per profile, weighted by work_authors row count.
  -- Require >=50% majority to drop profiles where stray contaminating raw_names
  -- compete with the canonical (the Rappard pollution case from Phase 1).
  SELECT author_id, raw_author_name
  FROM (
    SELECT author_id, raw_author_name, n_works,
           SUM(n_works) OVER (PARTITION BY author_id) AS total_works,
           ROW_NUMBER() OVER (PARTITION BY author_id
                              ORDER BY n_works DESC, raw_author_name ASC) AS rn
    FROM ra_counts
  )
  WHERE rn = 1 AND n_works * 2 >= total_works
),
parsed_dominant AS (
  -- Resolve dominant raw_name to (first, middle, last); rich-name filter
  SELECT rd.author_id,
         REGEXP_REPLACE(an.parsed_name.first,  '\.$', '') AS p_first,
         REGEXP_REPLACE(an.parsed_name.middle, '\.$', '') AS p_middle,
         REGEXP_REPLACE(an.parsed_name.last,   '\.$', '') AS p_last
  FROM ra_dominant rd
  JOIN openalex.authors.author_names an ON an.raw_author_name = rd.raw_author_name
  WHERE LENGTH(an.parsed_name.first)  >= 2
    AND LENGTH(an.parsed_name.middle) >= 3
    AND LENGTH(an.parsed_name.last)   >= 2
),
profile_with_dom AS (
  SELECT n.*, p.p_first, p.p_middle, p.p_last
  FROM no_orcid_profiles n
  JOIN parsed_dominant   p ON p.author_id = n.author_id
),
size2_groups AS (
  -- Size-exactly-2 groups by parsed (first, middle, last); sink cap on works_count
  SELECT p_first, p_middle, p_last
  FROM profile_with_dom
  GROUP BY p_first, p_middle, p_last
  HAVING COUNT(*) = 2 AND MAX(works_count) <= 5000
),
pair_profiles AS (
  SELECT pwd.*
  FROM profile_with_dom pwd
  JOIN size2_groups g
    ON pwd.p_first = g.p_first AND pwd.p_middle = g.p_middle AND pwd.p_last = g.p_last
),
pair_join AS (
  -- Pair (id_a, id_b) per group; carry institution arrays for inst-signal
  SELECT p1.p_first, p1.p_middle, p1.p_last,
         p1.author_id AS id_a, p2.author_id AS id_b,
         p1.lki_ids   AS lki_a, p2.lki_ids   AS lki_b,
         p1.aff_ids   AS aff_a, p2.aff_ids   AS aff_b
  FROM pair_profiles p1
  JOIN pair_profiles p2
    ON p1.p_first = p2.p_first AND p1.p_middle = p2.p_middle AND p1.p_last = p2.p_last
  WHERE p1.author_id < p2.author_id
),
inst_signal AS (
  SELECT id_a, id_b,
         (SIZE(ARRAY_INTERSECT(lki_a, lki_b)) > 0
          OR SIZE(ARRAY_INTERSECT(aff_a, aff_b)) > 0) AS inst_overlap
  FROM pair_join
),
coauthors_a AS (
  -- All coauthors of id_a (excluding id_a, id_b themselves)
  SELECT pj.id_a, pj.id_b, wa.author_id AS coauthor
  FROM pair_join pj
  JOIN openalex.works.work_authors wa_self ON wa_self.author_id = pj.id_a
  JOIN openalex.works.work_authors wa      ON wa.work_id = wa_self.work_id
  WHERE wa.author_id <> pj.id_a AND wa.author_id <> pj.id_b
),
coauthors_b AS (
  SELECT pj.id_a, pj.id_b, wa.author_id AS coauthor
  FROM pair_join pj
  JOIN openalex.works.work_authors wa_self ON wa_self.author_id = pj.id_b
  JOIN openalex.works.work_authors wa      ON wa.work_id = wa_self.work_id
  WHERE wa.author_id <> pj.id_a AND wa.author_id <> pj.id_b
),
coauthor_signal AS (
  -- Distinct coauthors who appear with both id_a (on some work) and id_b (on some work)
  SELECT a.id_a, a.id_b, COUNT(DISTINCT a.coauthor) AS n_shared_coauthors
  FROM coauthors_a a
  JOIN coauthors_b b
    ON a.id_a = b.id_a AND a.id_b = b.id_b AND a.coauthor = b.coauthor
  GROUP BY a.id_a, a.id_b
),
filtered_pairs AS (
  -- Keep pairs with at least one precision signal
  SELECT pj.id_a, pj.id_b, pj.p_first, pj.p_middle, pj.p_last,
         COALESCE(isig.inst_overlap, FALSE)   AS inst_overlap,
         COALESCE(csig.n_shared_coauthors, 0) AS n_shared_coauthors
  FROM pair_join pj
  LEFT JOIN inst_signal     isig ON isig.id_a = pj.id_a AND isig.id_b = pj.id_b
  LEFT JOIN coauthor_signal csig ON csig.id_a = pj.id_a AND csig.id_b = pj.id_b
  WHERE COALESCE(isig.inst_overlap, FALSE)
     OR COALESCE(csig.n_shared_coauthors, 0) >= 1
),
ranked AS (
  -- Two rows per pair (one per profile); rank within pair to pick winner
  SELECT fp.id_a, fp.id_b, fp.p_first, fp.p_middle, fp.p_last,
         fp.inst_overlap, fp.n_shared_coauthors,
         pp.author_id, pp.full_name, pp.display_name, pp.block_key,
         pp.works_count, pp.cited_by_count, pp.created_date,
         ROW_NUMBER() OVER (
           PARTITION BY fp.id_a, fp.id_b
           ORDER BY pp.works_count DESC, pp.created_date ASC, pp.author_id ASC
         ) AS rn
  FROM filtered_pairs fp
  JOIN pair_profiles pp
    ON pp.p_first = fp.p_first AND pp.p_middle = fp.p_middle AND pp.p_last = fp.p_last
   AND (pp.author_id = fp.id_a OR pp.author_id = fp.id_b)
),
winners AS (
  SELECT id_a, id_b, author_id AS winner_author_id FROM ranked WHERE rn = 1
)
SELECT
  r.id_a, r.id_b, r.p_first, r.p_middle, r.p_last,
  r.inst_overlap, r.n_shared_coauthors,
  r.author_id, r.full_name, r.display_name, r.block_key,
  r.works_count, r.cited_by_count, r.created_date,
  (r.rn = 1) AS is_winner,
  w.winner_author_id
FROM ranked r
JOIN winners w ON r.id_a = w.id_a AND r.id_b = w.id_b

### Cell 2: Validation statistics

In [ ]:
SELECT
  COUNT(DISTINCT id_a, id_b)                                               AS total_pairs,
  COUNT(*)                                                                 AS total_profiles,
  COUNT(CASE WHEN NOT is_winner THEN 1 END)                                AS losers_to_merge,
  COUNT(CASE WHEN     is_winner THEN 1 END)                                AS winners_kept,
  SUM(CASE WHEN NOT is_winner THEN works_count END)                        AS works_to_rewrite,
  COUNT(DISTINCT CASE WHEN inst_overlap THEN CONCAT(id_a, '-', id_b) END)  AS pairs_with_inst,
  COUNT(DISTINCT CASE WHEN n_shared_coauthors >= 1
                       THEN CONCAT(id_a, '-', id_b) END)                   AS pairs_with_coauthor,
  COUNT(DISTINCT CASE WHEN inst_overlap AND n_shared_coauthors >= 1
                       THEN CONCAT(id_a, '-', id_b) END)                   AS pairs_with_both,
  PERCENTILE_APPROX(works_count, ARRAY(0.5, 0.95, 0.99))                   AS profile_works_pctiles,
  PERCENTILE_APPROX(n_shared_coauthors, ARRAY(0.5, 0.95, 0.99))            AS coauthor_pctiles
FROM openalex.authors.merge_candidates_richname

### Cell 3: Spot-check sample (manual review)

In [ ]:
WITH sampled AS (
  SELECT DISTINCT id_a, id_b
  FROM openalex.authors.merge_candidates_richname
  ORDER BY RAND() LIMIT 30
)
SELECT c.id_a, c.id_b, c.p_first, c.p_middle, c.p_last,
       c.inst_overlap, c.n_shared_coauthors,
       c.author_id, c.full_name, c.block_key,
       c.works_count, c.cited_by_count,
       c.is_winner, c.winner_author_id
FROM openalex.authors.merge_candidates_richname c
JOIN sampled s ON c.id_a = s.id_a AND c.id_b = s.id_b
ORDER BY c.id_a, c.id_b, c.is_winner DESC, c.works_count DESC

### Cell 4: Create audit log

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_merge_log_richname AS
SELECT
  id_a, id_b, p_first, p_middle, p_last,
  inst_overlap, n_shared_coauthors,
  winner_author_id,
  author_id           AS loser_author_id,
  full_name           AS loser_full_name,
  works_count         AS loser_works_count,
  cited_by_count      AS loser_cited_by_count,
  created_date        AS loser_created_date,
  current_timestamp() AS merge_timestamp
FROM openalex.authors.merge_candidates_richname
WHERE NOT is_winner

### Cell 5: Snapshot affected work_ids (capture before MERGE rewrites author_id)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_merge_affected_works_richname AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_merge_log_richname log
  ON wa.author_id = log.loser_author_id

### Cell 6: Execute the merge — rewrite work_authors.author_id from losers to winners

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT loser_author_id, winner_author_id
  FROM openalex.authors.author_merge_log_richname
) AS source
ON target.author_id = source.loser_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = source.winner_author_id,
    target.updated_at = current_timestamp()

### Cell 7: Queue affected works for reprocessing by UpdateWorkAuthorships

*No DELETE step. Phase 1 reversed its DELETE so CreateAuthors could drain loser `works_count` to 0 on the next refresh; this notebook follows that pattern.*

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.author_merge_affected_works_richname

### Cell 8: Post-merge verification — losers should drain to 0 works after CreateAuthors

In [ ]:
WITH log AS (
  SELECT loser_author_id, loser_works_count, loser_full_name
  FROM openalex.authors.author_merge_log_richname
),
loser_state AS (
  SELECT l.loser_author_id, l.loser_works_count,
         a.works_count AS current_works_count
  FROM log l
  JOIN openalex.authors.openalex_authors a ON a.id = l.loser_author_id
)
SELECT
  COUNT(*)                                                       AS total_losers,
  COUNT(CASE WHEN current_works_count = 0 THEN 1 END)            AS losers_drained_to_zero,
  COUNT(CASE WHEN current_works_count > 0 THEN 1 END)            AS losers_still_nonzero,
  PERCENTILE_APPROX(current_works_count, ARRAY(0.5, 0.95, 0.99)) AS nonzero_pctiles
FROM loser_state